# ir_calendar_build_supplier_report — 供應商「財報＋法說整合報告」（HTML）

- 用途：對 `ir_calendar_parse_documents` 解析完的供應商財報 / 法說文件，依採購端模板（`template/`）產出一家公司一期的完整 HTML 報告：
  採購結論（燈號 / 分數 / 已證實事實 / 管理層展望 / 採購建議 / QBR 必問）→ 財報資訊（KPI、財報重點、實績表、事業群、五年 / 五季營收 × OPM 圖、十項指標圖、五年指標表）→ 法說會資訊（七章節、War Room、Watchlist）→ 資料來源與限制。
- 輸入（silver，全部由 `consume_batches` / `parse_documents` 維護，本 job 不改）：
  `s_{domain}_document_text`（文件全文）、`s_{domain}_document_file`（來源網址）、`s_{domain}_financials`（年度 + 季別財報數據）、
  `s_{domain}_summary`（爬蟲端法說彙整，只當參考）、`s_{domain}_company`、`s_{domain}_conference`（法說日期）。
- 輸出：
  - bronze `b_{domain}_supplier_report_llm`：每次 LLM 呼叫一列（原始回應 `payload`、tokens、狀態），append-only
  - gold `g_{domain}_supplier_report`：一家公司一期一列（`html` 全文、`llm_json` 結構化內容、燈號、版本、血緣），MERGE
  - Volume `<volume_root>/ir_calendar/reports/<分類小寫>/<公司代稱>/<代稱>_<期別>_supplier_report.html`（可直接開啟 / 寄送）
- 數字政策（使用者說明 四）：表格與圖表只用 `s_financials`；LLM 只寫文字、引用數字要標來源；缺值「未取得」、文件沒提「未提及」；沒有示意值模式。
- LLM：Azure Foundry `claude-sonnet-5`（同 meeting_record / sales_report_mail 的 secret `llm-api-keys/ms-foundry`）。一份報告 1～2 次呼叫、輸入最多 `max_total_chars` 字。
- 參數（widgets）：見 [c01] 與下表。
- 排程：接在 `ir_calendar_parse_documents` 之後（同 job 下一個 task），或每天一次；job cluster。`max_reports` 是費用閘門。
- 負責人 / 更新日期：（填）/ 2026-10-20
- 模板與使用者說明：`template/20261010_供應鏈供應商財務體質報告模板.html`、`template/USER_GUIDE.md`（含逐條對應與已知限制）。

## 一次執行做什麼

1. **工作清單** [c04] `select_work`：`s_document_text` 裡解析成功的文件，依 分類 / 公司 / 期別 篩選後 groupBy（公司, 期別）；`periods` 空 = 每家只取最新一期。
   扣掉 gold 已有「同 prompt 版、同版型、同一組文件 sha256」的報告（`force_rebuild` 不扣）。取前 `max_reports` 份。
2. **每份報告**（ThreadPool，`concurrency` 份併發）：
   1. [c09] 撈該公司的 financials 全部、該期的 summary / conference / 文件全文，collect 到 driver（都是小量）
   2. [c03] `build_facts`：換算單位（台股仟元 → 億、美股元 → 百萬美元）、五年 / 五季數列、實績表（本季 vs 去年同期 vs 上季）、KPI、指標可得性
   3. [c05] 依 `DOC_KIND_PRIORITY` 排文件、逐檔截尾、總量封頂 → user prompt；[c08] 呼叫 LLM；JSON 壞再問一次、截斷改精簡重打
   4. [c05] `normalize_report`：補齊固定鍵、收斂燈號 / 分數 / 四個採購面向
   5. [c07] `render_report`：套模板 CSS，[c06] 產 inline SVG 圖（標籤碰撞避讓、圖例在圖面上方）
   6. [c10] 寫 Volume HTML → gold MERGE；每次 LLM 呼叫都寫 bronze
3. 印每份結果與 tokens；有失敗且 `fail_on_error = true` 就 `raise`。

**重跑安全**：gold 以 (company_key, period) MERGE、Volume 同名覆蓋；bronze 只 append（保留每次呼叫）。同一期文件換新版本（sha256 變）會自動重做。

**只換版型**：改 [c06] / [c07] 後把 [c02] `TEMPLATE_VERSION` 升版，`render_only = true` 跑一次：不呼叫 LLM，用 gold 的 `llm_json` 重套。

## 參數

| widget | 預設 | 說明 |
|---|---|---|
| `catalog` / `schema` | `micenter` / `mi3_datahub_prod` | Delta table 所在；對照 `config/project.yml` |
| `domain` | `ir_calendar` | 表名中段 |
| `volume_root` | `/Volumes/micenter/…/unstructured_data_file` | 報告 HTML 落 `<volume_root>/ir_calendar/reports/` |
| `categories` | `SUPPLIER` | 逗號分隔分類；空 = 全部（客戶 / 同業也能產，但 prompt 是採購視角） |
| `company_keys` | 空 | 逗號分隔 `company_key`；空 = 範圍內全部 |
| `periods` | 空 | 逗號分隔 `yyyyQn`；空 = 每家最新一期 |
| `max_reports` | `5` | 單次最多幾份（費用閘門） |
| `max_doc_chars` / `max_total_chars` | `40000` / `120000` | 每份文件 / 全部文件餵 LLM 的字元上限 |
| `model` | `claude-sonnet-5` | Azure Foundry 模型名 |
| `prompt_version` | `1.0` | 要等於 [c05] `PROMPT_VERSION_IN_CELL` |
| `concurrency` | `2` | 併發 LLM 請求數 |
| `force_rebuild` | `false` | `true`：已有同版報告也重做（重新計費）；一定搭配 `company_keys` / `periods` |
| `render_only` | `false` | `true`：不呼叫 LLM，用 gold `llm_json` 重套版 |
| `dry_run` | `false` | `true`：只列工作清單、facts 摘要與 prompt 大小，不呼叫、不寫 |
| `fail_on_error` | `true` | 有失敗就讓 job 失敗 |
| `job_run_id` | 空 | job parameters 填 `{{job.run_id}}` |

## 維護者須知

| 要改什麼 | 改哪裡 |
|---|---|
| 報告章節 / 版型 | [c07]（CSS 來自 `template/`，不改規則只追加）；改完升 [c02] `TEMPLATE_VERSION`，`render_only` 重套 |
| 圖表樣式 / 標籤規則 | [c06]；`tests/test_ir_calendar_build_supplier_report.py` 有標籤不重疊的檢查 |
| LLM 要寫的內容 / 規則 | [c05] `SYSTEM_PROMPT` + 輸出結構；改了升 `PROMPT_VERSION_IN_CELL` 與 widget `prompt_version`；舊報告不會自動重做（`force_rebuild`） |
| 指標表多一項 / 補「未取得」的三項 | 先讓 `s_financials` 有欄位（爬蟲端 → `init_tables` ADD COLUMNS → `consume_batches` [c04]/[c06]），再改 [c02] `INDICATORS` 的 `source` |
| 金額顯示單位 | [c03] `money_scale` |
| 文件優先序 / 截尾 | [c02] `DOC_KIND_PRIORITY`、[c05] `pack_documents` |
| gold 多欄位 | `init_tables` [c19b] DDL `ADD COLUMNS` → [c10] `REPORT_SCHEMA` + `report_row` |
| 只想本機測邏輯 | [c02]～[c07] 沒有 I/O；`tests/test_ir_calendar_build_supplier_report.py` 用假 financials / 假 LLM JSON 跑 facts → render |

## 首次上線順序

1. 跑 `ir_calendar_init_tables`（含 [c19a] / [c19b] 兩張新表；既有表不受影響）
2. `dry_run = true`、`company_keys` 指一家有 financials 的台股供應商（例 `TPE:3481`）：看 facts 摘要、prompt 大小、落地路徑
3. `dry_run = false`、`max_reports = 1` 跑一次：打開 Volume 上的 HTML 檢查章節順序、圖表、未取得 / 未提及的呈現；對 gold 一列
4. 再試一家沒有 financials 的公司（TYO / KRX）確認「未取得」路徑
5. 放大 `max_reports`，排程接在 `parse_documents` 後


In [ ]:
# [c01] params
# 預設值取自 config/project.yml（catalog / schema）；job 執行時由 job parameters 覆蓋。
# widget 建立過後改 code 預設值不會生效，要重設先 dbutils.widgets.removeAll() 再跑本 cell。布林 / 數字都用字串填，這裡統一轉型。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")      # 表名中段；層級前綴 b_ / s_ / g_ 固定，見 docs/conventions.md 2.1
dbutils.widgets.text("volume_root", "/Volumes/micenter/mi3_datahub_prod/micenterfile_ext/unstructured_data_file")  # 報告 HTML 落 <volume_root>/ir_calendar/reports/
dbutils.widgets.text("categories", "SUPPLIER")    # 逗號分隔的分類代碼；預設只做供應商。空 = 全部分類
dbutils.widgets.text("company_keys", "")          # 逗號分隔的 company_key（TPE:3481,NYSE:GLW）；空 = 範圍內全部
dbutils.widgets.text("periods", "")               # 逗號分隔的曆年期別（2026Q2）；空 = 每家公司「有解析文件的最新一期」
dbutils.widgets.text("max_reports", "5")          # 費用閘門：單次最多產幾份報告（每份 1～2 次 LLM 呼叫、輸入可達 10 萬字）
dbutils.widgets.text("max_doc_chars", "40000")    # 每份文件最多餵 LLM 幾個字元（超過截尾並註明）
dbutils.widgets.text("max_total_chars", "120000") # 一份報告所有文件合計上限（依 [c02] DOC_KIND_PRIORITY 先放重要文件）
dbutils.widgets.text("model", "claude-sonnet-5")  # Azure Foundry 上的模型名（同 meeting_record / sales_report_mail）
dbutils.widgets.text("prompt_version", "1.0")     # 要等於 [c05] PROMPT_VERSION_IN_CELL；改 prompt 就升
dbutils.widgets.text("concurrency", "2")          # 同時打幾個 LLM 請求；輸入大，預設保守
dbutils.widgets.text("force_rebuild", "false")    # true：範圍內已有同版報告也重做（會重新計費）
dbutils.widgets.text("render_only", "false")      # true：不呼叫 LLM，用 gold 既有 llm_json 重新套版（改 [c06]/[c07] 版型時用）
dbutils.widgets.text("dry_run", "false")          # true：只列工作清單與 prompt 大小，不呼叫 LLM、不寫表、不寫 Volume
dbutils.widgets.text("fail_on_error", "true")     # true：本次有失敗的報告就讓 job 失敗（通知才會發）
dbutils.widgets.text("job_run_id", "")            # job parameters 填 {{job.run_id}}


def _flag(name: str) -> bool:
    return dbutils.widgets.get(name).strip().lower() == "true"


def _csv(name: str) -> list[str]:
    return [x.strip() for x in dbutils.widgets.get(name).split(",") if x.strip()]


settings = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "volume_root": dbutils.widgets.get("volume_root").strip().rstrip("/"),
    "categories": _csv("categories"),
    "company_keys": _csv("company_keys"),
    "periods": _csv("periods"),
    "max_reports": int(dbutils.widgets.get("max_reports")),
    "max_doc_chars": int(dbutils.widgets.get("max_doc_chars")),
    "max_total_chars": int(dbutils.widgets.get("max_total_chars")),
    "model": dbutils.widgets.get("model").strip(),
    "prompt_version": dbutils.widgets.get("prompt_version").strip(),
    "concurrency": max(1, int(dbutils.widgets.get("concurrency"))),
    "force_rebuild": _flag("force_rebuild"),
    "render_only": _flag("render_only"),
    "dry_run": _flag("dry_run"),
    "fail_on_error": _flag("fail_on_error"),
    "job_run_id": dbutils.widgets.get("job_run_id").strip() or None,
}
assert settings["catalog"] and settings["schema"] and settings["domain"], "catalog / schema / domain 不可為空"
assert settings["volume_root"].startswith("/Volumes/"), "volume_root 必須是 UC Volume 路徑"
assert settings["max_reports"] >= 1 and settings["max_doc_chars"] >= 1000, "max_reports ≥ 1；max_doc_chars ≥ 1000"
assert settings["concurrency"] <= 8, "concurrency ≤ 8"
assert settings["model"] and settings["prompt_version"], "model / prompt_version 不可為空"
if settings["force_rebuild"] and not (settings["company_keys"] or settings["periods"]):
    print("!!! force_rebuild 沒有搭配 company_keys / periods：範圍內最多 max_reports 份報告會重新計費")
print(settings)


In [ ]:
# [c02] imports
# 共用 import 與常數。本 cell 與 [c03]～[c07] 不碰 spark 全域變數 / dbutils / 檔案系統，可被 tests/ 載入。
import html
import json
import math
import re
import time
from datetime import datetime, timezone

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

UTC = timezone.utc  # noqa: UP017 - 本機測試仍是 Python 3.10，不用 datetime.UTC

# 表命名：<層級前綴><domain>_<短名>，層級前綴固定（docs/conventions.md 2.1）
LAYER_PREFIX = {"bronze": "b_", "silver": "s_", "gold": "g_"}
SRC_COMPANY, SRC_DOC_TEXT, SRC_DOC_FILE = "company", "document_text", "document_file"
SRC_SUMMARY, SRC_FINANCIALS, SRC_CONFERENCE = "summary", "financials", "conference"
LLM_TABLE = "supplier_report_llm"     # bronze：每次 LLM 呼叫一列（原始回應 payload），append-only
REPORT_TABLE = "supplier_report"      # gold：一家公司一期一列（最新版 HTML + 結構化 JSON），MERGE
REPORT_DIR = "ir_calendar/reports"    # <volume_root>/ir_calendar/reports/<分類小寫>/<公司代稱>/<代稱>_<期別>_supplier_report.html

TEMPLATE_VERSION = "20261010"         # template/20261010_供應鏈供應商財務體質報告模板.html；[c07] 版型改了就升
DOC_STATUS_OK = ("success", "partial")
# 餵 LLM 的文件順序：逐字稿 / 簡報最能回答法說七章節，先放；財報本體最長，放最後、受 max_total_chars 截尾
DOC_KIND_PRIORITY = ["Earnings_Transcript", "Presentation", "Press_Release", "Financial_Statements"]

# 近五年指標表的 11 項（使用者說明 二）。source = s_ir_calendar_financials 欄位；None = 目前沒有來源 → 畫面「未取得」、不畫圖。
# kind：money（金額，依 money_scale 換算）/ pct（%）/ ratio（倍）/ eps（元）/ days（天）
INDICATORS = [
    {"key": "revenue", "label": "營收", "kind": "money", "source": "revenue"},
    {"key": "gross_margin", "label": "毛利率", "kind": "pct", "source": "gross_margin"},
    {"key": "pretax_margin", "label": "淨利率（稅前）", "kind": "pct", "source": "pretax_margin"},   # [c03] 由 pretax_income / revenue 算
    {"key": "net_margin", "label": "稅後淨利率", "kind": "pct", "source": "net_margin"},
    {"key": "eps", "label": "EPS", "kind": "eps", "source": "eps"},
    {"key": "operating_cash_flow", "label": "營業現金流", "kind": "money", "source": "operating_cash_flow"},
    {"key": "debt_ratio", "label": "負債比", "kind": "pct", "source": "debt_ratio"},
    {"key": "current_ratio", "label": "流動比", "kind": "ratio", "source": "current_ratio"},
    {"key": "quick_ratio", "label": "速動比", "kind": "ratio", "source": None},        # 無存貨欄位
    {"key": "ar_days", "label": "應收天數", "kind": "days", "source": None},           # 無應收帳款欄位
    {"key": "ap_days", "label": "應付天數", "kind": "days", "source": None},           # 無應付帳款 / 銷貨成本（台股）欄位
]
INDICATOR_BY_KEY = {i["key"]: i for i in INDICATORS}
N_YEARS, N_QUARTERS = 5, 5

# 燈號（模板 🟢 / 🟡 / 🔴）
LIGHTS = {
    "green": {"emoji": "🟢", "zh": "綠燈", "action": "維持往來", "css": "green"},
    "amber": {"emoji": "🟡", "zh": "黃燈", "action": "加強監控", "css": "amber"},
    "red": {"emoji": "🔴", "zh": "紅燈", "action": "降載", "css": "red"},
}
VERDICTS = {"positive": "🟢 Positive", "stable": "🟢 Stable", "watch": "🟡 Watch", "risk": "🔴 Risk"}
ACTION_AREAS = ["供應穩定性", "成本與議價", "付款條件", "替代料或驗證風險"]
NOT_MENTIONED, NOT_AVAILABLE = "未提及", "未取得"


In [ ]:
# [c03] pure_facts
# 純 Python：把 silver 撈出來的小量列（一家公司的 financials 年度 + 季別、法說彙整、文件清單）整理成報告要用的 facts。
# 數字只來自 s_ir_calendar_financials；算不出來就是 None，畫面顯示「未取得」，絕不補值（使用者說明 四）。
# 這裡刻意用 list[dict] 不用 DataFrame：一家公司最多幾十列，collect 到 driver 比 Spark 好測、好讀。


def money_scale(currency: str | None, unit: str | None) -> dict:
    """原始金額 → 報告顯示單位。回 {divisor, label, prefix, note}。
    台股 MOPS 仟元 → 億；美股 SEC 元 → 百萬美元；其他幣別以「原始單位 → 億 / 百萬」保守處理並在 note 說明。"""
    cur = (currency or "").upper()
    base = {"thousand": 1e3, "one": 1.0, "million": 1e6}.get((unit or "one").lower(), 1.0)
    if cur == "TWD":
        return {"divisor": 1e8 / base, "label": "億", "prefix": "NT$", "note": "台股 MOPS 仟元換算為億元"}
    if cur == "USD":
        return {"divisor": 1e6 / base, "label": "百萬", "prefix": "US$", "note": "美股 SEC 元換算為百萬美元"}
    if cur in ("JPY", "KRW", "CNY", "HKD"):
        return {"divisor": 1e8 / base, "label": "億", "prefix": cur, "note": f"{cur} 換算為億 {cur}"}
    return {"divisor": 1e6 / base, "label": "百萬", "prefix": cur or "", "note": f"{cur or '未知幣別'} 換算為百萬"}


def to_num(v) -> float | None:
    if v is None:
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(f) or math.isinf(f) else f


def fmt_value(v: float | None, kind: str, *, signed: bool = False) -> str:
    """畫面用的數字字串；None → 未取得。money 已換算；pct 帶 %；ratio 帶 x；eps 兩位小數；days 整數。"""
    if v is None:
        return NOT_AVAILABLE
    if kind == "pct":
        return f"{v:+.2f}%" if signed else f"{v:.2f}%"
    if kind == "ratio":
        return f"{v:.2f}x"
    if kind == "days":
        return f"{v:.0f}"
    if kind == "eps":
        return f"{v:+.2f}" if signed else f"{v:.2f}"
    return f"{v:+,.2f}" if signed else f"{v:,.2f}"


def period_parts(period: str) -> tuple[int, int | None]:
    """'2026Q2' → (2026, 2)；'2025FY' → (2025, None)。格式不合回 (0, None)。"""
    m = re.fullmatch(r"(\d{4})(?:Q([1-4])|FY)", period or "")
    if not m:
        return 0, None
    return int(m.group(1)), (int(m.group(2)) if m.group(2) else None)


def period_label(period: str) -> str:
    y, q = period_parts(period)
    return f"{y} Q{q}" if q else str(y)


def prev_quarter(period: str, n: int = 1) -> str:
    y, q = period_parts(period)
    if not q:
        return ""
    idx = y * 4 + (q - 1) - n
    return f"{idx // 4}Q{idx % 4 + 1}"


def indicator_values(row: dict, scale: dict) -> dict:
    """一列 financials → {indicator_key: 顯示單位的值或 None}。pretax_margin 由 pretax_income / revenue 推算。"""
    out = {}
    for ind in INDICATORS:
        src = ind["source"]
        if src is None:
            out[ind["key"]] = None
        elif src == "pretax_margin":
            pre, rev = to_num(row.get("pretax_income")), to_num(row.get("revenue"))
            out[ind["key"]] = (pre / rev * 100.0) if (pre is not None and rev) else None
        elif ind["kind"] == "money":
            v = to_num(row.get(src))
            out[ind["key"]] = (v / scale["divisor"]) if v is not None else None
        else:
            out[ind["key"]] = to_num(row.get(src))
    # 實績表 / KPI 另外要的金額欄
    for col in ("net_income", "operating_income", "total_assets", "total_liabilities", "capex"):
        v = to_num(row.get(col))
        out[col] = (v / scale["divisor"]) if v is not None else None
    out["operating_margin"] = to_num(row.get("operating_margin"))
    return out


def _latest_rows(rows: list[dict]) -> dict[str, dict]:
    """同 period 多列（理論上 silver 已 MERGE 唯一）取 seq 最大。回 {period: row}。"""
    out: dict[str, dict] = {}
    for r in rows:
        p = r.get("period")
        if not p:
            continue
        if p not in out or (r.get("seq") or 0) >= (out[p].get("seq") or 0):
            out[p] = r
    return out


def build_annual_series(fin_rows: list[dict], scale: dict, *, n: int = N_YEARS) -> list[dict]:
    """年度列（period_type = annual）最近 n 年，由舊到新。每項 {period, label, values}。"""
    annual = _latest_rows([r for r in fin_rows if r.get("period_type") == "annual"])
    keys = sorted(annual, key=lambda p: period_parts(p)[0])[-n:]
    return [{"period": p, "label": period_label(p), "values": indicator_values(annual[p], scale)} for p in keys]


def build_quarter_series(fin_rows: list[dict], scale: dict, period: str, *, n: int = N_QUARTERS) -> list[dict]:
    """季別列到 period 為止（含）的最近 n 季，由舊到新；中間缺季就缺，不補。"""
    quarterly = _latest_rows([r for r in fin_rows if r.get("period_type") == "quarterly"])
    y, q = period_parts(period)
    if not q:
        return []
    wanted = [prev_quarter(period, i) for i in range(n - 1, -1, -1)]
    return [{"period": p, "label": period_label(p), "values": indicator_values(quarterly[p], scale)}
            for p in wanted if p in quarterly]


def build_actuals(fin_rows: list[dict], scale: dict, period: str) -> dict:
    """財報實績：本季 vs 去年同期 vs 上季。回 {columns:[{period,label,values}|None ×3], metrics:[...]}。"""
    quarterly = _latest_rows([r for r in fin_rows if r.get("period_type") == "quarterly"])
    cols = []
    for p, tag in ((period, "本季"), (prev_quarter(period, 4), "去年同期"), (prev_quarter(period, 1), "上季")):
        r = quarterly.get(p)
        cols.append({"period": p, "label": f"{tag} {period_label(p)}" if p else tag,
                     "values": indicator_values(r, scale) if r else None})
    metrics = [("revenue", f"營收（{scale['label']}）", "money"), ("gross_margin", "毛利率", "pct"),
               ("operating_margin", "營業利益率", "pct"), ("net_income", f"稅後淨利（{scale['label']}）", "money"),
               ("net_margin", "稅後淨利率", "pct"), ("eps", "EPS（元）", "eps"),
               ("operating_cash_flow", f"營業現金流（{scale['label']}）", "money")]
    return {"columns": cols, "metrics": [{"key": k, "label": lb, "kind": kd} for k, lb, kd in metrics]}


def build_kpis(actuals: dict, scale: dict) -> list[dict]:
    """頁首四格 KPI（模板一）：營收 + YoY、毛利率 + 去年同期、營業利益率、流動比率。"""
    cur, yoy, _ = actuals["columns"]
    cv = cur["values"] or {}
    yv = yoy["values"] or {}
    rev, rev_y = cv.get("revenue"), yv.get("revenue")
    yoy_pct = ((rev - rev_y) / abs(rev_y) * 100.0) if (rev is not None and rev_y) else None
    return [
        {"label": f"{period_label(cur['period'])} 營收", "value": f"{scale['prefix']} {fmt_value(rev, 'money')} {scale['label']}" if rev is not None else NOT_AVAILABLE,
         "note": f"YoY {fmt_value(yoy_pct, 'pct', signed=True)}" if yoy_pct is not None else "YoY 未取得",
         "tone": "red" if (yoy_pct is not None and yoy_pct < 0) else ("green" if yoy_pct is not None else "")},
        {"label": f"{period_label(cur['period'])} 毛利率", "value": fmt_value(cv.get("gross_margin"), "pct"),
         "note": f"去年同期：{fmt_value(yv.get('gross_margin'), 'pct')}", "tone": ""},
        {"label": f"{period_label(cur['period'])} 營業利益率", "value": fmt_value(cv.get("operating_margin"), "pct", signed=True),
         "note": "本業獲利" if (cv.get("operating_margin") or 0) > 0 else ("本業虧損" if cv.get("operating_margin") is not None else NOT_AVAILABLE),
         "tone": "green" if (cv.get("operating_margin") or 0) > 0 else ("red" if cv.get("operating_margin") is not None else "")},
        {"label": f"{period_label(cur['period'])} 流動比率", "value": fmt_value(cv.get("current_ratio"), "ratio"),
         "note": "短期償債緩衝" if (cv.get("current_ratio") or 0) >= 1 else ("低於 1x，留意短期償債" if cv.get("current_ratio") is not None else NOT_AVAILABLE),
         "tone": "green" if (cv.get("current_ratio") or 0) >= 1 else ("red" if cv.get("current_ratio") is not None else "")},
    ]


def indicator_availability(annual: list[dict]) -> dict[str, bool]:
    """十一項各自在五年數列裡有沒有至少一個值。"""
    return {ind["key"]: any(s["values"].get(ind["key"]) is not None for s in annual) for ind in INDICATORS}


def build_facts(company: dict, period: str, fin_rows: list[dict], summary: dict | None,
                conference: dict | None, docs: list[dict]) -> dict:
    """報告資料包：[c05] 拿去組 prompt、[c07] 拿去套版。docs 每項至少有 file_name / doc_kind / sha256 / source_url / text_chars。"""
    sample = next((r for r in fin_rows if r.get("currency")), None)
    scale = money_scale(sample.get("currency"), sample.get("unit")) if sample else money_scale(None, None)
    annual = build_annual_series(fin_rows, scale)
    quarterly = build_quarter_series(fin_rows, scale, period)
    actuals = build_actuals(fin_rows, scale, period)
    sources = sorted({u for r in fin_rows for u in (r.get("source_urls") or []) if u})
    return {
        "company": {k: company.get(k) for k in ("company_key", "company_name", "english_name", "stock_code",
                                                  "market", "category", "company_slug", "industry", "ir_url")},
        "period": period,
        "period_label": period_label(period),
        "conference_date": (conference or {}).get("conference_date"),
        "scale": scale,
        "has_financials": bool(fin_rows),
        "financial_source": sorted({r.get("source") for r in fin_rows if r.get("source")}),
        "financial_source_urls": sources,
        "annual": annual,
        "quarterly": quarterly,
        "actuals": actuals,
        "kpis": build_kpis(actuals, scale),
        "indicator_avail": indicator_availability(annual),
        "summary": summary or {},
        "docs": [{k: d.get(k) for k in ("file_name", "doc_kind", "sha256", "source_url", "text_chars", "volume_path", "fiscal_label")}
                 for d in docs],
    }


def facts_to_prompt_text(facts: dict) -> str:
    """結構化數據的文字版（給 LLM 看，讓它引用時有依據、不用自己算）。"""
    c, scale = facts["company"], facts["scale"]
    lines = [f"公司：{c.get('company_name')}（{c.get('english_name') or ''}；{c.get('company_key')}；分類 {c.get('category')}；"
             f"產業 {c.get('industry') or '未知'}）", f"報告期別：{facts['period']}（曆年季）",
             f"法說日期：{facts.get('conference_date') or '未知'}"]
    if not facts["has_financials"]:
        lines.append("【結構化財報數據】未取得：此公司市場目前沒有結構化財報來源。表格與圖表會顯示「未取得」；"
                     "你在文字中引用任何財報數字時，必須來自下方文件並標明文件名。")
    else:
        lines.append(f"【結構化財報數據】來源 {'/'.join(facts['financial_source'])}；幣別 {scale['prefix']}；"
                     f"金額已換算為「{scale['label']}」（{scale['note']}）；比率為 %。")
        cols = [i for i in INDICATORS if facts["indicator_avail"].get(i["key"])]
        head = "期別 | " + " | ".join(i["label"] for i in cols) + " | 營業利益率"
        lines += ["年度（近五年）：", head]
        for s in facts["annual"]:
            lines.append(f"{s['period']} | " + " | ".join(fmt_value(s["values"].get(i["key"]), i["kind"]) for i in cols)
                         + f" | {fmt_value(s['values'].get('operating_margin'), 'pct')}")
        lines += ["季別（近五季）：", head]
        for s in facts["quarterly"]:
            lines.append(f"{s['period']} | " + " | ".join(fmt_value(s["values"].get(i["key"]), i["kind"]) for i in cols)
                         + f" | {fmt_value(s['values'].get('operating_margin'), 'pct')}")
        missing = [i["label"] for i in INDICATORS if not facts["indicator_avail"].get(i["key"])]
        if missing:
            lines.append("未取得的指標（不要自行推算或引用）：" + "、".join(missing))
    sm = facts.get("summary") or {}
    if sm:
        lines.append("【爬蟲端法說彙整（LLM 先前產出，僅供參考，以文件原文為準）】")
        for k, lb in (("core_points", "重點"), ("key_numbers", "關鍵數字"), ("risks", "風險"),
                      ("positive_signals", "正面訊號"), ("watch_items", "觀察事項"), ("open_questions", "待釐清")):
            vals = sm.get(k) or []
            if vals:
                lines.append(f"{lb}：" + "；".join(str(v) for v in vals))
        if sm.get("guidance"):
            lines.append(f"展望：{sm['guidance']}")
        for mv in sm.get("management_views") or []:
            if isinstance(mv, dict) and mv.get("view"):
                lines.append(f"經營層觀點［{mv.get('topic')}］：{mv['view']}")
    return "\n".join(lines)


In [ ]:
# [c04] spark_transforms
# DataFrame -> DataFrame 純函式：工作清單（哪些 公司 × 期別 要產報告）。無 I/O，可被 tests/ 載入。
WORK_COLS = ["company_key", "company_slug", "category", "period", "doc_count", "doc_kinds", "docs_digest"]


def latest_per_key(df: DataFrame, keys: list[str], order_col: str = "updated_at") -> DataFrame:
    """同鍵多列取 order_col 最大的那列（silver 理論上唯一，這裡保險）。"""
    w = Window.partitionBy(*keys).orderBy(F.col(order_col).desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")


def candidate_reports(df_text: DataFrame, *, categories: list[str], company_keys: list[str],
                      periods: list[str]) -> DataFrame:
    """s_document_text → 每個 (公司, 期別) 一列：有幾份解析成功的文件、哪些種類、文件版本摘要 docs_digest。
    periods 空 = 每家公司只取「有文件的最新一期」（期別字串 yyyyQn 可直接比大小）。"""
    df = df_text.filter(F.col("parse_status").isin(*DOC_STATUS_OK) & F.col("period").rlike(r"^\d{4}Q[1-4]$"))
    if categories:
        df = df.filter(F.col("category").isin(*categories))
    if company_keys:
        df = df.filter(F.col("company_key").isin(*company_keys))
    if periods:
        df = df.filter(F.col("period").isin(*periods))
    grouped = (df.groupBy("company_key", "company_slug", "category", "period")
               .agg(F.count("*").alias("doc_count"),
                    F.array_sort(F.collect_set("doc_kind")).alias("doc_kinds"),
                    F.sha2(F.concat_ws(",", F.array_sort(F.collect_set("sha256"))), 256).alias("docs_digest")))
    if not periods:
        w = Window.partitionBy("company_key").orderBy(F.col("period").desc())
        grouped = grouped.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
    return grouped.select(*WORK_COLS)


def select_work(df_text: DataFrame, df_report: DataFrame, *, categories: list[str], company_keys: list[str],
                periods: list[str], prompt_version: str, template_version: str, force: bool) -> DataFrame:
    """候選扣掉 gold 已有「同 prompt 版、同版型、同一組文件」的報告（force = True 不扣）。
    文件有新版本（sha256 變）或 prompt / 版型升版 → docs_digest / 版本不同 → 重做。"""
    cand = candidate_reports(df_text, categories=categories, company_keys=company_keys, periods=periods)
    if force:
        return cand
    done = (df_report.filter((F.col("prompt_version") == F.lit(prompt_version))
                             & (F.col("template_version") == F.lit(template_version)))
            .select("company_key", "period", "docs_digest"))
    return cand.join(done, ["company_key", "period", "docs_digest"], "left_anti").select(*WORK_COLS)


In [ ]:
# [c05] prompt
# LLM 只做「讀文件、寫文字段落」：採購結論、財報重點、事業群、法說七章節、War Room、Watchlist。
# 數字表格與圖表由 [c03] / [c06] 從結構化數據產生，LLM 不得編數字（使用者說明 四）。prompt 內嵌成常數；改這個 cell 就升 [c01] prompt_version。
PROMPT_VERSION_IN_CELL = "1.0"   # [c11] 會比對 settings["prompt_version"]，不一致就停
LLM_MAX_TOKENS = 12000
TRUNCATED_MARK = "\n…（本文件超過長度上限，以下截尾）"

SYSTEM_PROMPT = """你是 AUO（友達光電）採購部門的供應商分析師。輸入是一家供應商某一季的公開資料：結構化財報數據（程式已算好）、爬蟲端先前的法說彙整、以及該季的財報 / 法說文件全文（PDF / HTML 解析後的 markdown）。
你的工作是寫出「財報＋法說整合報告」的文字內容，輸出 **單一 JSON 物件**，不加任何說明、不用 markdown 圍欄。

## 寫作規則
1. 繁體中文、採購主管可直接讀的商務語氣；每條 ≤ 80 字；用詞具體（產品線、產能、期程、金額），不寫空話。
2. 嚴格區分三種內容：**已證實事實**（文件或財報數據明載）、**管理層展望**（公司自己的預期 / 指引）、**採購建議**（你的判讀）。不得混寫。
3. 採購結論聚焦四個面向：供應穩定性、成本與議價、付款條件、替代料或驗證風險。**不得**把公司整體財務變動直接推論為個別料號的漲價或降價理由；只能寫成「議價時可留意 / 需向供應商確認」。
4. 引用數字只能來自「結構化財報數據」或文件原文，並在句尾標來源：（財報數據）或（文件名）。不得自行推算、不得補缺值。結構化數據標「未取得」的指標不要引用。
5. 文件沒提到的章節，對應欄位給空陣列 []（畫面會顯示「未提及」）。**不得**推測與 AUO 的合作關係、供應項目或 Micro LED 相關性；只有文件明確提到才寫。
6. 「與 AUO 業務及 Micro LED 相關議題」：只收文件中明確提到 AUO / 友達、面板 / 顯示器客戶、Micro LED、Mini LED 的段落；沒有就 []。
7. Q&A 重點：只有逐字稿或簡報有 Q&A 才填；每題 q 一句、a 一到兩句。
8. 爬蟲端法說彙整只是參考；與文件原文衝突時以原文為準。
9. 燈號與分數是「採購管理用示範分數」：green（維持往來）/ amber（加強監控）/ red（降載）；分數 0～100，依獲利趨勢、現金流、負債結構、供貨能力綜合判斷；在 one_liner 用 2～4 句說明判讀依據。

## 輸出 JSON 結構（鍵名固定，全部必填；沒有內容給 [] 或 ""）
{
  "procurement_conclusion": {
    "light": "green | amber | red",
    "score": 0-100 的整數,
    "one_liner": "2～4 句：營收與獲利趨勢、資產負債緩衝、本業獲利與現金流、主要風險",
    "confirmed_facts": ["已證實事實 …（來源）"],
    "management_outlook": ["管理層展望 …（來源）"],
    "procurement_actions": [
      {"area": "供應穩定性", "advice": "…"},
      {"area": "成本與議價", "advice": "…"},
      {"area": "付款條件", "advice": "…"},
      {"area": "替代料或驗證風險", "advice": "…"}
    ],
    "qbr_questions": ["下次 QBR 必問 …（3～5 題）"]
  },
  "financial_highlights": ["財報重點 …（來源）"],
  "segment_performance": [{"segment": "事業群 / 產品線名", "result": "該期表現（含數字與來源）", "comment": "一句解讀"}],
  "conference": {
    "business_lines": ["事業體／產品線重點"],
    "capacity": ["擴產／產能變化"],
    "profitability": ["獲利能力變化"],
    "market_outlook": ["未來市場預測"],
    "investment_ma_org": ["投資／M&A／組織整改"],
    "auo_microled": ["與 AUO 業務及 Micro LED 相關議題"],
    "qa": [{"q": "問題", "a": "回答重點"}]
  },
  "war_room": [
    {"item": "評估項目（新事業成長 / 本業 / 現金流 / 財務結構 / 關鍵零件供應 / 供貨能力 …）",
     "management_view": "管理層訊息", "financial_check": "財務驗證（引用數據或寫 未取得）",
     "verdict": "positive | stable | watch | risk"}
  ],
  "positive_signals": ["正向訊號"],
  "watch_items": ["觀察事項"],
  "sources_used": ["實際引用到的文件名"],
  "caveats": ["口徑限制、文件缺漏、你無法判斷的地方"]
}"""

RETRY_HINT = ("你上一輪的輸出無法解析為 JSON：{err}。請只輸出一個合法 JSON 物件（鍵名照規格、字串內的引號要跳脫、"
              "不要 markdown 圍欄、不要前後說明）。")
COMPACT_HINT = ("\n\n（上一輪輸出超過長度上限被截斷。請精簡：每個陣列最多 6 條、每條 ≤ 50 字、qa 最多 6 題，其餘規則不變。）")

REPORT_LIST_KEYS = ["financial_highlights", "segment_performance", "war_room", "positive_signals", "watch_items",
                    "sources_used", "caveats"]
CONF_KEYS = ["business_lines", "capacity", "profitability", "market_outlook", "investment_ma_org", "auo_microled", "qa"]
PC_LIST_KEYS = ["confirmed_facts", "management_outlook", "procurement_actions", "qbr_questions"]


def pack_documents(docs: list[dict], *, max_doc_chars: int, max_total_chars: int) -> tuple[list[dict], list[str]]:
    """依 DOC_KIND_PRIORITY 排序、逐檔截尾、總量封頂。回 (放進 prompt 的文件 [{file_name, doc_kind, text, truncated}], 被略過的檔名)。"""
    order = {k: i for i, k in enumerate(DOC_KIND_PRIORITY)}
    ranked = sorted(docs, key=lambda d: (order.get(d.get("doc_kind"), len(order)), d.get("file_name") or ""))
    out, skipped, used = [], [], 0
    for d in ranked:
        text = (d.get("text_md") or "").strip()
        if not text:
            skipped.append(d.get("file_name") or "?")
            continue
        room = min(max_doc_chars, max_total_chars - used)
        if room < 2000:
            skipped.append(d.get("file_name") or "?")
            continue
        truncated = len(text) > room
        body = text[:room] + TRUNCATED_MARK if truncated else text
        out.append({"file_name": d.get("file_name"), "doc_kind": d.get("doc_kind"), "text": body, "truncated": truncated})
        used += len(body)
    return out, skipped


def build_user_prompt(facts: dict, packed_docs: list[dict], skipped: list[str]) -> str:
    parts = ["# 結構化資料", facts_to_prompt_text(facts), "", "# 文件全文（markdown；表格以 | 分隔）"]
    if not packed_docs:
        parts.append("（本期沒有可用的文件全文）")
    for d in packed_docs:
        parts += [f"\n## 文件：{d['file_name']}（{d['doc_kind']}{'；已截尾' if d['truncated'] else ''}）", d["text"]]
    if skipped:
        parts.append("\n（因長度上限未放入的文件：" + "、".join(skipped) + "）")
    parts.append("\n請依 system 規格輸出 JSON。")
    return "\n".join(parts)


def parse_llm_json(text: str | None) -> tuple[dict | None, str | None]:
    """容忍 ```json 圍欄與前後雜訊：取第一個 { 到最後一個 }。回 (obj, error)。"""
    if not text or not text.strip():
        return None, "empty output"
    s = text.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    i, j = s.find("{"), s.rfind("}")
    if i < 0 or j <= i:
        return None, "no JSON object found"
    try:
        obj = json.loads(s[i:j + 1])
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e}"
    if not isinstance(obj, dict):
        return None, "top level is not an object"
    return obj, None


def _str_list(v) -> list[str]:
    if isinstance(v, str):
        return [v.strip()] if v.strip() else []
    if not isinstance(v, list):
        return []
    return [str(x).strip() for x in v if x is not None and str(x).strip()]


def _dict_list(v, keys: list[str]) -> list[dict]:
    if not isinstance(v, list):
        return []
    out = []
    for x in v:
        if isinstance(x, dict):
            out.append({k: str(x.get(k) or "").strip() for k in keys})
        elif isinstance(x, str) and x.strip():
            out.append({keys[0]: x.strip(), **{k: "" for k in keys[1:]}})
    return [d for d in out if any(d.values())]


def normalize_report(obj: dict | None) -> dict:
    """LLM 輸出 → 固定形狀（缺鍵補空、型別修正、燈號 / 分數收斂），[c07] 才不用到處判 None。"""
    obj = obj or {}
    pc = obj.get("procurement_conclusion") if isinstance(obj.get("procurement_conclusion"), dict) else {}
    light = str(pc.get("light") or "").strip().lower()
    light = light if light in LIGHTS else "amber"
    try:
        score = int(round(float(pc.get("score"))))
    except (TypeError, ValueError):
        score = None
    score = min(100, max(0, score)) if score is not None else None
    actions = _dict_list(pc.get("procurement_actions"), ["area", "advice"])
    by_area = {a["area"]: a["advice"] for a in actions if a["advice"]}
    actions = [{"area": a, "advice": by_area.get(a, "")} for a in ACTION_AREAS] + \
              [a for a in actions if a["area"] not in ACTION_AREAS and a["advice"]]
    conf = obj.get("conference") if isinstance(obj.get("conference"), dict) else {}
    war = _dict_list(obj.get("war_room"), ["item", "management_view", "financial_check", "verdict"])
    for w in war:
        w["verdict"] = w["verdict"].lower() if w["verdict"].lower() in VERDICTS else "watch"
    return {
        "procurement_conclusion": {
            "light": light, "score": score, "one_liner": str(pc.get("one_liner") or "").strip(),
            "confirmed_facts": _str_list(pc.get("confirmed_facts")),
            "management_outlook": _str_list(pc.get("management_outlook")),
            "procurement_actions": actions,
            "qbr_questions": _str_list(pc.get("qbr_questions")),
        },
        "financial_highlights": _str_list(obj.get("financial_highlights")),
        "segment_performance": _dict_list(obj.get("segment_performance"), ["segment", "result", "comment"]),
        "conference": {**{k: _str_list(conf.get(k)) for k in CONF_KEYS if k != "qa"},
                       "qa": _dict_list(conf.get("qa"), ["q", "a"])},
        "war_room": war,
        "positive_signals": _str_list(obj.get("positive_signals")),
        "watch_items": _str_list(obj.get("watch_items")),
        "sources_used": _str_list(obj.get("sources_used")),
        "caveats": _str_list(obj.get("caveats")),
    }


In [ ]:
# [c06] charts_svg
# 純 Python 產 inline SVG：淺藍柱（營收 / 指標）＋ 紫色折線（OPM / 趨勢）。不用 matplotlib（cluster 無中文字型會變豆腐）、
# 不用 JS / CDN（報告要能離線單獨開啟、列印）。版面規則照使用者說明 二：營收數字放柱內中間偏上、所有標籤不互相遮蓋、
# 不被折線 / 圖例遮蓋（圖例放圖面上方獨立列；標籤框碰撞自動避讓；文字加白邊 halo，折線穿過也讀得到）。
# chart_layout() 先算幾何（含每個標籤的框，tests 用它驗「不重疊」），chart_svg() 只負責畫。
CHART_W, CHART_H = 1100, 380
PLOT = {"l": 72, "r": 72, "t": 48, "b": 44}
BAR_FILL, BAR_STROKE = "#9ec5f0", "#6ea4dc"      # 淺藍
LINE_COLOR = "#7b3fa0"                           # 紫
GRID, AXIS_TXT, MUTED = "#dbe4ef", "#65738a", "#9aa7b8"
FONT = "Inter,'Noto Sans TC','Microsoft JhengHei',sans-serif"
LABEL_FS, AXIS_FS = 12, 11
LABEL_H = 14


def _nice_step(span: float, divisions: int = 4) -> float:
    raw = span / divisions if span > 0 else 1.0
    mag = 10 ** math.floor(math.log10(raw))
    for m in (1, 2, 2.5, 5, 10):
        if raw <= m * mag:
            return m * mag
    return 10 * mag


def _axis(values: list[float | None], *, headroom: float = 0.18) -> dict:
    """含 0 的軸：回 {min, max, step, ticks}。上方留 headroom 給標籤。"""
    vals = [v for v in values if v is not None]
    lo, hi = min(0.0, *vals) if vals else 0.0, max(0.0, *vals) if vals else 1.0
    if hi == lo:
        hi = lo + 1.0
    span = hi - lo
    hi += span * headroom
    if lo < 0:
        lo -= span * 0.08
    step = _nice_step(hi - lo)
    lo = math.floor(lo / step) * step
    hi = math.ceil(hi / step) * step
    ticks = []
    t = lo
    while t <= hi + step * 1e-9:
        ticks.append(round(t, 10))
        t += step
    return {"min": lo, "max": hi, "step": step, "ticks": ticks}


def _text_w(s: str, fs: float = LABEL_FS) -> float:
    """估文字寬：ASCII 0.58em、CJK 1.0em。只用來算標籤框，不求精準。"""
    return sum((1.0 if ord(ch) > 0x2E7F else 0.58) * fs for ch in s) + 6


def _box(cx: float, y_baseline: float, text: str, fs: float = LABEL_FS) -> dict:
    w = _text_w(text, fs)
    return {"x": cx - w / 2, "y": y_baseline - fs, "w": w, "h": LABEL_H, "text": text, "cx": cx, "ty": y_baseline}


def _overlap(a: dict, b: dict, pad: float = 2.0) -> bool:
    return not (a["x"] + a["w"] + pad <= b["x"] or b["x"] + b["w"] + pad <= a["x"]
                or a["y"] + a["h"] + pad <= b["y"] or b["y"] + b["h"] + pad <= a["y"])


def chart_layout(labels: list[str], bars: list[float | None], line: list[float | None] | None, *,
                 bar_kind: str, line_kind: str = "pct", line_labels: bool = True,
                 shared_axis: bool = False) -> dict:
    """算幾何。bars / line 長度 = labels；None = 缺值（柱不畫、寫「未取得」；折線斷開）。
    shared_axis = True：折線用柱的軸（單一指標圖：柱 + 趨勢線同一數列）。"""
    n = max(1, len(labels))
    px, py = PLOT["l"], PLOT["t"]
    pw, ph = CHART_W - PLOT["l"] - PLOT["r"], CHART_H - PLOT["t"] - PLOT["b"]
    slot = pw / n
    bw = slot * 0.5
    ax_b = _axis(bars)
    ax_l = ax_b if (shared_axis or not line) else _axis(line)

    def yb(v: float) -> float:
        return py + (ax_b["max"] - v) / (ax_b["max"] - ax_b["min"]) * ph

    def yl(v: float) -> float:
        return py + (ax_l["max"] - v) / (ax_l["max"] - ax_l["min"]) * ph

    y0 = yb(0.0)
    bar_rects, bar_labels, missing = [], [], []
    for i, v in enumerate(bars):
        cx = px + slot * (i + 0.5)
        if v is None:
            missing.append(_box(cx, y0 - 6, NOT_AVAILABLE, AXIS_FS))
            continue
        top, bottom = (yb(v), y0) if v >= 0 else (y0, yb(v))
        bar_rects.append({"x": cx - bw / 2, "y": top, "w": bw, "h": max(0.0, bottom - top), "v": v})
        text = fmt_value(v, bar_kind)
        h = bottom - top
        if v >= 0:
            ty = top + 16 if h >= 26 else top - 5          # 柱內中間偏上；柱太矮放柱頂上方
        else:
            ty = bottom + 14                               # 負值放柱底下方
        bar_labels.append(_box(cx, ty, text))

    pts, line_labels_out = [], []
    if line:
        for i, v in enumerate(line):
            if v is None:
                pts.append(None)
                continue
            cx = px + slot * (i + 0.5)
            pts.append({"x": cx, "y": yl(v), "v": v})
    taken = bar_labels + missing
    if line and line_labels:
        for p in pts:
            if p is None:
                continue
            text = fmt_value(p["v"], line_kind, signed=(line_kind == "pct"))
            cands = [p["y"] - 9, p["y"] + 19]
            placed = None
            for ty in cands:
                b = _box(p["x"], ty, text)
                if b["y"] >= py - 2 and b["y"] + b["h"] <= CHART_H - 4 and not any(_overlap(b, t) for t in taken):
                    placed = b
                    break
            if placed is None:                              # 兩個位置都撞：往上找空位
                ty = p["y"] - 9
                for _ in range(24):
                    b = _box(p["x"], ty, text)
                    if not any(_overlap(b, t) for t in taken):
                        placed = b
                        break
                    ty -= 5
                placed = placed or _box(p["x"], p["y"] - 9, text)
            taken.append(placed)
            line_labels_out.append(placed)

    return {
        "w": CHART_W, "h": CHART_H, "plot": {"x": px, "y": py, "w": pw, "h": ph}, "slot": slot, "y0": y0,
        "axis_bar": {**ax_b, "ticks_y": [yb(t) for t in ax_b["ticks"]]},
        "axis_line": None if (not line or shared_axis) else {**ax_l, "ticks_y": [yl(t) for t in ax_l["ticks"]]},
        "labels": labels, "bar_kind": bar_kind, "line_kind": line_kind,
        "bars": bar_rects, "bar_labels": bar_labels, "missing": missing,
        "points": pts, "line_labels": line_labels_out,
    }


def overlapping_labels(layout: dict) -> list[tuple[str, str]]:
    """tests 用：回所有互相重疊的標籤文字對（應為空）。"""
    boxes = layout["bar_labels"] + layout["line_labels"] + layout["missing"]
    out = []
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            if _overlap(boxes[i], boxes[j]):
                out.append((boxes[i]["text"], boxes[j]["text"]))
    return out


def _svg_text(x: float, y: float, text: str, *, fs: float = LABEL_FS, fill: str = "#172033", anchor: str = "middle",
              weight: str = "600", halo: bool = True) -> str:
    style = f"font-family:{FONT};font-size:{fs}px;font-weight:{weight};"
    if halo:
        style += "paint-order:stroke;stroke:#ffffff;stroke-width:3px;stroke-linejoin:round;"
    return (f'<text x="{x:.1f}" y="{y:.1f}" text-anchor="{anchor}" fill="{fill}" style="{style}">'
            f"{html.escape(text)}</text>")


def chart_svg(layout: dict, *, bar_name: str, line_name: str | None, bar_unit: str, line_unit: str = "%",
              title: str = "") -> str:
    """layout → <svg>。圖例在最上一列（y≈20），圖面從 PLOT.t 開始，互不重疊。"""
    L = layout
    p = L["plot"]
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {L["w"]} {L["h"]}" width="100%" '
           f'role="img" aria-label="{html.escape(title or bar_name)}" style="display:block;max-width:100%;height:auto">']
    # 圖例列
    lx = p["x"]
    out.append(f'<rect x="{lx}" y="12" width="14" height="14" rx="2" fill="{BAR_FILL}" stroke="{BAR_STROKE}"/>')
    out.append(_svg_text(lx + 20, 24, f"{bar_name}（{bar_unit}）", fs=AXIS_FS, fill=AXIS_TXT, anchor="start", weight="500", halo=False))
    if line_name:
        lx2 = lx + 20 + _text_w(f"{bar_name}（{bar_unit}）", AXIS_FS) + 18
        out.append(f'<line x1="{lx2}" y1="19" x2="{lx2 + 22}" y2="19" stroke="{LINE_COLOR}" stroke-width="2.5"/>')
        out.append(f'<circle cx="{lx2 + 11}" cy="19" r="3.2" fill="{LINE_COLOR}"/>')
        out.append(_svg_text(lx2 + 28, 24, f"{line_name}（{line_unit}）", fs=AXIS_FS, fill=AXIS_TXT, anchor="start", weight="500", halo=False))
    # 格線 + 左軸
    ab = L["axis_bar"]
    for t, y in zip(ab["ticks"], ab["ticks_y"], strict=True):
        out.append(f'<line x1="{p["x"]}" y1="{y:.1f}" x2="{p["x"] + p["w"]}" y2="{y:.1f}" stroke="{GRID}" stroke-width="1"/>')
        out.append(_svg_text(p["x"] - 8, y + 4, _tick_text(t, L["bar_kind"]), fs=AXIS_FS, fill=AXIS_TXT, anchor="end", weight="500", halo=False))
    # 右軸（折線）
    if L["axis_line"]:
        al = L["axis_line"]
        for t, y in zip(al["ticks"], al["ticks_y"], strict=True):
            out.append(_svg_text(p["x"] + p["w"] + 8, y + 4, _tick_text(t, L["line_kind"]), fs=AXIS_FS, fill=LINE_COLOR, anchor="start", weight="500", halo=False))
    # 零線
    out.append(f'<line x1="{p["x"]}" y1="{L["y0"]:.1f}" x2="{p["x"] + p["w"]}" y2="{L["y0"]:.1f}" stroke="#8fa2b8" stroke-width="1.2"/>')
    # 柱
    for r in L["bars"]:
        out.append(f'<rect x="{r["x"]:.1f}" y="{r["y"]:.1f}" width="{r["w"]:.1f}" height="{r["h"]:.1f}" rx="3" '
                   f'fill="{BAR_FILL}" stroke="{BAR_STROKE}" stroke-width="1"/>')
    # 折線 + 點
    pts = L["points"]
    if pts:
        segs, cur = [], []
        for pt in pts:
            if pt is None:
                if cur:
                    segs.append(cur)
                cur = []
            else:
                cur.append(pt)
        if cur:
            segs.append(cur)
        for seg in segs:
            if len(seg) >= 2:
                d = " ".join(f"{'M' if i == 0 else 'L'}{q['x']:.1f},{q['y']:.1f}" for i, q in enumerate(seg))
                out.append(f'<path d="{d}" fill="none" stroke="{LINE_COLOR}" stroke-width="2.6" stroke-linejoin="round"/>')
            for q in seg:
                out.append(f'<circle cx="{q["x"]:.1f}" cy="{q["y"]:.1f}" r="4" fill="{LINE_COLOR}" stroke="#fff" stroke-width="1.5"/>')
    # 標籤（最後畫，疊在線與柱之上）
    for b in L["bar_labels"]:
        out.append(_svg_text(b["cx"], b["ty"], b["text"], fill="#0b1f3a"))
    for b in L["missing"]:
        out.append(_svg_text(b["cx"], b["ty"], b["text"], fs=AXIS_FS, fill=MUTED, weight="500"))
    for b in L["line_labels"]:
        out.append(_svg_text(b["cx"], b["ty"], b["text"], fill=LINE_COLOR))
    # X 軸類別
    for i, lb in enumerate(L["labels"]):
        cx = p["x"] + L["slot"] * (i + 0.5)
        out.append(_svg_text(cx, L["h"] - 14, lb, fs=AXIS_FS, fill=AXIS_TXT, weight="500", halo=False))
    out.append("</svg>")
    return "\n".join(out)


def _tick_text(t: float, kind: str) -> str:
    if kind == "pct":
        return f"{t:g}%"
    if kind == "ratio":
        return f"{t:g}x"
    return f"{t:,.0f}" if abs(t) >= 10 else f"{t:g}"


def svg_revenue_opm(series: list[dict], scale: dict, *, title: str) -> str:
    """營收（柱）× 營業利益率（折線）。series = [c03] build_annual_series / build_quarter_series 的輸出。"""
    labels = [s["label"] for s in series]
    bars = [s["values"].get("revenue") for s in series]
    line = [s["values"].get("operating_margin") for s in series]
    layout = chart_layout(labels, bars, line, bar_kind="money", line_kind="pct")
    return chart_svg(layout, bar_name="營收", line_name="營業利益率 OPM", bar_unit=f"{scale['prefix']} {scale['label']}",
                     line_unit="%", title=title)


def svg_indicator(series: list[dict], key: str, scale: dict) -> str:
    """單一指標五年圖：淺藍柱 + 同數列紫色趨勢線（共用軸、折線不另標數字，避免重複）。"""
    ind = INDICATOR_BY_KEY[key]
    labels = [s["label"] for s in series]
    vals = [s["values"].get(key) for s in series]
    layout = chart_layout(labels, vals, vals, bar_kind=ind["kind"], line_kind=ind["kind"], line_labels=False, shared_axis=True)
    unit = {"money": f"{scale['prefix']} {scale['label']}", "pct": "%", "ratio": "倍", "eps": "元", "days": "天"}[ind["kind"]]
    return chart_svg(layout, bar_name=ind["label"], line_name="趨勢", bar_unit=unit, line_unit="同左", title=ind["label"])


In [ ]:
# [c07] render_html
# facts（[c03]）+ LLM 結構化輸出（[c05] normalize_report）→ 完整可單獨開啟的 HTML。
# 版型：template/20261010_供應鏈供應商財務體質報告模板.html 的 <style> 原樣內嵌（TEMPLATE_CSS），沿用 hero / card / table / callout / 燈號環；
# 章節順序依使用者說明：hero → 給 AUO 採購主管的結論 → 財報資訊 → 法說會資訊 → 資料來源與限制（template/USER_GUIDE.md §2）。
# 所有文字經 html.escape；數字只從 facts 來；空清單一律印「未提及」，缺值印「未取得」。
TEMPLATE_CSS = """:root{--navy:#0b1f3a;--blue:#1f5aa6;--cyan:#20a4b7;--ink:#172033;--muted:#65738a;--bg:#f3f6fa;--card:#fff;--green:#16835d;--amber:#bd7300;--red:#c33c3c;--line:#dbe4ef;--fill:#fff3b0}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);font-family:Inter,"Noto Sans TC","Microsoft JhengHei",sans-serif;color:var(--ink);line-height:1.55}
.hero{background:linear-gradient(120deg,var(--navy),#143d72 65%,#16798f);color:#fff;padding:38px 5vw 30px}
.hero h1{margin:0;font-size:clamp(26px,4vw,42px)}
.hero p{margin:8px 0 0;color:#d8e8f7}
.badge{display:inline-block;padding:5px 10px;border-radius:999px;background:#ffffff20;border:1px solid #ffffff50;font-size:12px;margin-bottom:12px}
.wrap{max-width:1220px;margin:auto;padding:24px}
.grid{display:grid;gap:16px}
.kpis{grid-template-columns:repeat(4,1fr);margin-top:-44px}
.card{background:#fff;border:1px solid var(--line);border-radius:16px;padding:20px;box-shadow:0 8px 26px #1a3b6610}
.label{font-size:13px;color:var(--muted)}
.value{font-size:28px;font-weight:800;margin:4px 0}
.note,.small{font-size:12px;color:var(--muted)}
.red{color:var(--red)}
.green{color:var(--green)}
.amber{color:var(--amber)}
h2{font-size:22px;margin:6px 0 14px}
h3{font-size:16px;margin:0 0 8px}
.two{grid-template-columns:1.2fr .8fr;margin-top:16px}
.mt{margin-top:16px}
table{width:100%;border-collapse:collapse;font-size:14px}
th,td{padding:10px 8px;border-bottom:1px solid var(--line);text-align:right}
th:first-child,td:first-child{text-align:left}
th{color:var(--muted);font-weight:600;background:#f8fafc}
.matrix th,.matrix td{text-align:left}
.matrix th:last-child,.matrix td:last-child{text-align:center;white-space:nowrap}
.callout{border-left:4px solid var(--amber);background:#fff8e8;padding:14px 16px;border-radius:8px;margin-top:14px}
ul,ol{padding-left:20px;margin:8px 0}
li{margin:5px 0}
.score{display:flex;align-items:center;gap:18px}
.ring{width:104px;height:104px;border-radius:50%;display:grid;place-items:center;background:conic-gradient(var(--amber) 0 50%,#e8eef5 50%);position:relative;flex:0 0 auto}
.ring:after{content:"";position:absolute;inset:12px;border-radius:50%;background:#fff}
.ring b{position:relative;z-index:1;font-size:25px}
footer{padding:18px 0 42px;color:var(--muted);font-size:12px;text-align:center}
a{color:var(--blue)}
mark{background:var(--fill);color:inherit;padding:0 3px;border-radius:3px}
@media(max-width:850px){.kpis,.two{grid-template-columns:1fr 1fr}}
@media(max-width:600px){.kpis,.two{grid-template-columns:1fr}.kpis{margin-top:0}}
@media print{body{background:#fff}.hero{padding:24px}.wrap{padding:12px}.card{box-shadow:none;break-inside:avoid}mark{background:none}}
/* 本 job 追加（不改模板既有規則）：章節錨點、單欄圖表列、區塊標題 */
.section-title{font-size:24px;margin:28px 0 12px;color:var(--navy);border-left:6px solid var(--blue);padding-left:12px}
.chart{margin:10px 0 4px}
.chart-row{margin-top:16px}
.chart-row h3{margin-bottom:4px}
.tbl-scroll{overflow-x:auto}
.na{color:var(--muted)}
.pill{display:inline-block;padding:2px 8px;border-radius:999px;font-size:12px;background:#eef3f9;color:var(--muted);margin-left:6px}
"""

SECTION_TITLES = {"conclusion": "給 AUO 採購主管的結論", "financial": "財報資訊", "conference": "法說會資訊",
                  "sources": "資料來源與限制"}
CONF_SECTIONS = [("business_lines", "事業體／產品線重點"), ("capacity", "擴產／產能變化"), ("profitability", "獲利能力變化"),
                 ("market_outlook", "未來市場預測"), ("investment_ma_org", "投資／M&A／組織整改"),
                 ("auo_microled", "與 AUO 業務及 Micro LED 相關議題"), ("qa", "Q&A 重點")]


def esc(s) -> str:
    return html.escape("" if s is None else str(s), quote=True)


def ul(items: list[str], empty: str = NOT_MENTIONED) -> str:
    if not items:
        return f'<ul><li class="na">{esc(empty)}</li></ul>'
    return "<ul>" + "".join(f"<li>{esc(x)}</li>" for x in items) + "</ul>"


def ol(items: list[str], empty: str = NOT_MENTIONED) -> str:
    if not items:
        return f'<ol><li class="na">{esc(empty)}</li></ol>'
    return "<ol>" + "".join(f"<li>{esc(x)}</li>" for x in items) + "</ol>"


def _tone_cls(v: float | None, *, good_positive: bool = True) -> str:
    if v is None:
        return "na"
    return ("green" if v > 0 else "red" if v < 0 else "") if good_positive else ""


def render_hero(facts: dict, llm: dict, *, generated_on: str, version: str) -> str:
    c = facts["company"]
    return f"""<header class="hero">
  <div class="badge">Supplier Financial Health｜財報＋法說整合報告｜採購視角</div>
  <h1>{esc(c.get('company_name'))}（{esc(c.get('stock_code') or c.get('company_key'))}）財報＋法說整合報告</h1>
  <p>資料基準：{esc(facts['period_label'])} 公開資訊｜法說日期：{esc(facts.get('conference_date') or '未取得')}｜產出日期：{esc(generated_on)}｜版本：{esc(version)}</p>
</header>"""


def render_kpis(facts: dict) -> str:
    cards = []
    for k in facts["kpis"]:
        tone = f" {k['tone']}" if k["tone"] else ""
        cards.append(f'<div class="card"><div class="label">{esc(k["label"])}</div><div class="value">{esc(k["value"])}</div>'
                     f'<div class="note{tone}">{esc(k["note"])}</div></div>')
    return '<section class="grid kpis">' + "".join(cards) + "</section>"


def render_conclusion(facts: dict, llm: dict) -> str:
    pc = llm["procurement_conclusion"]
    light = LIGHTS[pc["light"]]
    score = pc["score"]
    pct = score if score is not None else 0
    ring = f'<div class="ring" style="background:conic-gradient(var(--{light["css"]}) 0 {pct}%,#e8eef5 {pct}%)"><b>{esc(score if score is not None else "–")}</b></div>'
    actions = "".join(f"<li><b>{esc(a['area'])}：</b>{esc(a['advice'] or NOT_MENTIONED)}</li>" for a in pc["procurement_actions"])
    return f"""<h2 class="section-title" id="conclusion">{SECTION_TITLES['conclusion']}</h2>
<section class="grid two">
  <article class="card">
    <h2>管理層重點結論</h2>
    <div class="score">
      {ring}
      <div>
        <h3>暫定燈號：{light['emoji']} {light['zh']}（{light['action']}）</h3>
        <div class="small">分數 {esc(score if score is not None else NOT_AVAILABLE)} / 100 ※ 採購管理用示範分數，非信用評等或投資建議。</div>
      </div>
    </div>
    <h3 class="mt">一句話判讀</h3>
    <div class="callout" style="margin-top:6px">{esc(pc['one_liner'] or NOT_MENTIONED)}</div>
    <h3 class="mt">已證實事實</h3>{ul(pc['confirmed_facts'])}
    <h3 class="mt">管理層展望</h3>{ul(pc['management_outlook'])}
  </article>
  <article class="card">
    <h2>採購建議</h2>
    <ul>{actions}</ul>
    <div class="small">※ 不以公司整體財務變動直接推論個別料號的價格；議價前以料號層級成本與供應商確認。</div>
    <h2 class="mt">下次 QBR 必問</h2>
    {ol(pc['qbr_questions'])}
  </article>
</section>"""


def render_actuals(facts: dict) -> str:
    a = facts["actuals"]
    head = "".join(f"<th>{esc(c['label'])}</th>" for c in a["columns"])
    rows = []
    for m in a["metrics"]:
        cells = []
        for c in a["columns"]:
            v = (c["values"] or {}).get(m["key"]) if c["values"] else None
            cells.append(f"<td>{esc(fmt_value(v, m['kind'], signed=m['kind'] in ('pct', 'eps') and m['key'] != 'gross_margin'))}</td>")
        rows.append(f"<tr><td>{esc(m['label'])}</td>{''.join(cells)}</tr>")
    note = "" if facts["has_financials"] else f'<div class="callout"><b>未取得：</b>此公司目前沒有結構化財報來源（{esc(facts["company"].get("market"))} 市場），實績表、指標表與圖表皆無法產生；財報重點僅依文件文字整理。</div>'
    return f"""<section class="card mt">
  <h2>財報實績（{esc(facts['period_label'])}）</h2>
  <div class="tbl-scroll"><table><thead><tr><th>項目</th>{head}</tr></thead><tbody>{''.join(rows)}</tbody></table></div>
  <div class="small">金額單位：{esc(facts['scale']['prefix'])} {esc(facts['scale']['label'])}（{esc(facts['scale']['note'])}）；比率為 %；季度數值不可直接與全年比較。</div>
  {note}
</section>"""


def render_segments(llm: dict) -> str:
    segs = llm["segment_performance"]
    if not segs:
        body = f'<div class="na">{NOT_MENTIONED}（文件未提供事業群 / 產品線分部資料）</div>'
    else:
        rows = "".join(f"<tr><td>{esc(s['segment'])}</td><td>{esc(s['result'])}</td><td>{esc(s['comment'])}</td></tr>" for s in segs)
        body = f'<table class="matrix"><thead><tr><th>事業群 / 產品線</th><th>本期表現</th><th>解讀</th></tr></thead><tbody>{rows}</tbody></table>'
    return f'<section class="card mt"><h2>事業群財務表現</h2>{body}</section>'


def render_indicator_table(facts: dict) -> str:
    annual = facts["annual"]
    head = "".join(f"<th>{esc(i['label'])}</th>" for i in INDICATORS)
    if not annual:
        rows = f'<tr><td class="na" colspan="{len(INDICATORS) + 1}">{NOT_AVAILABLE}</td></tr>'
    else:
        rows = "".join(f"<tr><td>{esc(s['label'])}</td>" + "".join(f"<td>{esc(fmt_value(s['values'].get(i['key']), i['kind']))}</td>" for i in INDICATORS) + "</tr>"
                       for s in annual)
    missing = [i["label"] for i in INDICATORS if not facts["indicator_avail"].get(i["key"])]
    note = f'<div class="small">未取得：{esc("、".join(missing))}（目前財報數據來源沒有對應欄位，不推算、不畫圖）</div>' if missing else ""
    return f"""<section class="card mt">
  <h2>近五年財務指標</h2>
  <div class="tbl-scroll"><table><thead><tr><th>年度</th>{head}</tr></thead><tbody>{rows}</tbody></table></div>
  <div class="small">營收 / 營業現金流單位：{esc(facts['scale']['prefix'])} {esc(facts['scale']['label'])}；EPS 為元；比率為 %；流動比 / 速動比為倍。</div>
  {note}
</section>"""


def render_charts(facts: dict) -> str:
    scale = facts["scale"]
    parts = ['<section class="card mt"><h2>五年、五季「營收 × OPM」趨勢</h2>']
    for title, series in (("近五年（年度）", facts["annual"]), ("近五季（季度）", facts["quarterly"])):
        parts.append(f'<div class="chart-row"><h3>{esc(title)}</h3>')
        if series:
            parts.append(f'<div class="chart">{svg_revenue_opm(series, scale, title=title)}</div>')
        else:
            parts.append(f'<div class="na">{NOT_AVAILABLE}</div>')
        parts.append("</div>")
    parts.append('<div class="small">淺藍柱：營收；紫線：營業利益率（OPM）。年度與季度口徑不同，兩圖不可直接比較。</div></section>')
    parts.append('<section class="card mt"><h2>近五年指標趨勢（其餘十項，每項一列）</h2>')
    for ind in INDICATORS:
        if ind["key"] == "revenue":
            continue                                   # 上方已有五年營收圖，不重畫
        parts.append(f'<div class="chart-row" data-indicator="{ind["key"]}"><h3>{esc(ind["label"])}</h3>')
        if facts["indicator_avail"].get(ind["key"]):
            parts.append(f'<div class="chart">{svg_indicator(facts["annual"], ind["key"], scale)}</div>')
        else:
            parts.append(f'<div class="na">{NOT_AVAILABLE}（財報數據來源無此欄位）</div>')
        parts.append("</div>")
    parts.append("</section>")
    return "\n".join(parts)


def render_financial(facts: dict, llm: dict) -> str:
    return "\n".join([
        f'<h2 class="section-title" id="financial">{SECTION_TITLES["financial"]}</h2>',
        f'<section class="card mt"><h2>財報重點</h2>{ul(llm["financial_highlights"])}</section>',
        render_actuals(facts), render_segments(llm), render_charts(facts), render_indicator_table(facts),
    ])


def render_conference(facts: dict, llm: dict) -> str:
    conf = llm["conference"]
    blocks = []
    for key, title in CONF_SECTIONS:
        if key == "qa":
            qa = conf["qa"]
            body = (f'<ul class="na"><li>{NOT_MENTIONED}</li></ul>' if not qa else
                    "<ul>" + "".join(f"<li><b>Q：</b>{esc(x['q'])}<br><b>A：</b>{esc(x['a'])}</li>" for x in qa) + "</ul>")
        else:
            body = ul(conf[key])
        blocks.append(f"<h3 class=\"mt\">{esc(title)}</h3>{body}")
    war = llm["war_room"]
    war_rows = ("".join(f"<tr><td>{esc(w['item'])}</td><td>{esc(w['management_view'])}</td><td>{esc(w['financial_check'])}</td>"
                        f"<td>{VERDICTS[w['verdict']]}</td></tr>" for w in war)
                or f'<tr><td class="na" colspan="4">{NOT_MENTIONED}</td></tr>')
    sm = facts.get("summary") or {}
    crawler_note = ""
    if sm.get("guidance") or sm.get("core_points"):
        crawler_note = f'<div class="small mt">爬蟲端法說彙整（參考）：{esc(sm.get("guidance") or "；".join(sm.get("core_points") or []))}</div>'
    return f"""<h2 class="section-title" id="conference">{SECTION_TITLES['conference']}</h2>
<section class="card mt">
  <h2>法說會重點（{esc(facts['period_label'])}）</h2>
  {''.join(blocks)}
  {crawler_note}
</section>
<section class="card mt">
  <h2>Executive War Room｜法說會展望 × 財務驗證 × 採購風險矩陣</h2>
  <table class="matrix"><thead><tr><th>評估項目</th><th>管理層觀點</th><th>財務驗證</th><th>AUO 判讀</th></tr></thead><tbody>{war_rows}</tbody></table>
</section>
<section class="card mt">
  <h2>Supplier Watchlist｜採購主管版</h2>
  <h3>🟢 Positive Signals</h3>{ul(llm['positive_signals'])}
  <h3>🟡 Watch Items</h3>{ul(llm['watch_items'])}
</section>"""


def render_sources(facts: dict, llm: dict) -> str:
    items = []
    for d in facts["docs"]:
        name = esc(d.get("file_name"))
        link = f'<a href="{esc(d["source_url"])}">{name}</a>' if d.get("source_url") else name
        items.append(f"<li>{link} — {esc(d.get('doc_kind'))}；解析字元 {esc(d.get('text_chars') or 0)}</li>")
    if facts["has_financials"]:
        src = "、".join(facts["financial_source"]) or "s_ir_calendar_financials"
        urls = "".join(f' <a href="{esc(u)}">[{i + 1}]</a>' for i, u in enumerate(facts["financial_source_urls"][:6]))
        items.append(f"<li>結構化財報數據：{esc(src)}{urls} — 用於 KPI、財報實績、近五年指標表與全部圖表</li>")
    sm = facts.get("summary") or {}
    for s in sm.get("sources") or []:
        if isinstance(s, dict) and s.get("url"):
            items.append(f'<li><a href="{esc(s["url"])}">{esc(s.get("title") or s["url"])}</a> — 爬蟲端法說彙整引用</li>')
    caveats = llm["caveats"]
    limits = [f"金額口徑：{facts['scale']['note']}；比率由財報數據來源算好（%）；年度與季度口徑不同，不可直接比較。"]
    missing = [i["label"] for i in INDICATORS if not facts["indicator_avail"].get(i["key"])]
    if missing:
        limits.append("未取得指標：" + "、".join(missing) + "（來源無對應欄位，不推算）。")
    if not facts["has_financials"]:
        limits.append("此公司無結構化財報來源，所有數字表格與圖表為「未取得」。")
    limits.append("文字段落由 LLM 依上列文件整理，數字引用以文件原文與公開資訊觀測站 / 公司合併財報附註為準；正式決策前請再核對。")
    return f"""<h2 class="section-title" id="sources">{SECTION_TITLES['sources']}</h2>
<section class="card mt">
  <h3>資料來源</h3>
  <ul>{''.join(items) or '<li class="na">無</li>'}</ul>
  <h3 class="mt">LLM 實際引用</h3>{ul(llm['sources_used'], empty='未標示')}
  <h3 class="mt">限制</h3>
  <div class="callout" style="margin-top:6px">{'<br>'.join(esc(x) for x in limits + caveats)}</div>
</section>"""


def render_report(facts: dict, llm: dict, *, generated_on: str, version: str) -> str:
    """完整 HTML。generated_on = 'YYYY-MM-DD'；version 例 'v1 / prompt 1.0 / template 20261010'。"""
    c = facts["company"]
    title = f"{c.get('company_name')}（{c.get('stock_code') or c.get('company_key')}）{facts['period_label']} 財報＋法說整合報告"
    return f"""<!doctype html>
<html lang="zh-Hant-TW">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{esc(title)}</title>
<meta name="generator" content="ir_calendar_build_supplier_report {esc(version)}">
<style>
{TEMPLATE_CSS}</style>
</head>
<body>
{render_hero(facts, llm, generated_on=generated_on, version=version)}
<main class="wrap">
{render_kpis(facts)}
{render_conclusion(facts, llm)}
{render_financial(facts, llm)}
{render_conference(facts, llm)}
{render_sources(facts, llm)}
</main>
<footer>Generated for procurement review｜{esc(generated_on)}｜{esc(version)}｜AUO 採購內部使用，非投資建議</footer>
</body>
</html>
"""


def report_file_name(facts: dict) -> str:
    slug = facts["company"].get("company_slug") or facts["company"].get("company_key", "unknown").replace(":", "_")
    return f"{slug}_{facts['period']}_supplier_report.html"


In [ ]:
# [c08] llm_client
# LLM 接線：Azure Foundry 的 Anthropic Messages 端點，與 meeting_extracted_w1_t2_extract [c06] / sales_report_mail 同一組 secret / URL。
# requests 是 DBR 內建，不 %pip install。thinking 關閉、只要 JSON；system 開 prompt caching（同一份 system prompt 多家公司共用）。
# 不送 temperature：claude-sonnet-5 回 400「temperature is deprecated」（2026-10-16 實測）。
# 重試：HTTP 408 / 429 / 5xx / 連線錯誤指數退避 3 次；其他 4xx 直接回 error。JSON 失敗的重試在 [c11]。
import requests

FOUNDRY_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry")
HTTP_TIMEOUT_S = 300          # 輸入可達 10 萬字，給長一點


def call_llm(messages: list[dict], system_prompt: str, model: str, *, cache: bool = True,
             max_tokens: int = LLM_MAX_TOKENS) -> dict:
    """回 {"text", "stop_reason", "usage": {input, output, cache_read, cache_write}, "latency_s", "error"}；
    error 非 None 代表端點層失敗（已重試），text 為 None。"""
    system = [{"type": "text", "text": system_prompt}]
    if cache:
        system[0]["cache_control"] = {"type": "ephemeral"}
    body = {"model": model, "max_tokens": max_tokens, "system": system, "messages": messages,
            "thinking": {"type": "disabled"}}
    headers = {"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"}
    t0 = time.time()
    err = None
    for attempt in range(3):
        try:
            resp = requests.post(FOUNDRY_URL, headers=headers, json=body, timeout=HTTP_TIMEOUT_S)
            if resp.status_code in (408, 429) or resp.status_code >= 500:
                err = f"HTTP {resp.status_code}: {resp.text[:200]}"
            elif resp.status_code >= 400:
                return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2),
                        "error": f"HTTP {resp.status_code}: {resp.text[:300]}"}
            else:
                data = resp.json()
                text = "\n".join(b.get("text", "") for b in data.get("content", []) if b.get("type") == "text")
                u = data.get("usage") or {}
                usage = {"input": u.get("input_tokens", 0) or 0, "output": u.get("output_tokens", 0) or 0,
                         "cache_read": u.get("cache_read_input_tokens", 0) or 0,
                         "cache_write": u.get("cache_creation_input_tokens", 0) or 0}
                return {"text": text, "stop_reason": data.get("stop_reason"), "usage": usage,
                        "latency_s": round(time.time() - t0, 2), "error": None}
        except (requests.RequestException, ValueError) as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
        time.sleep(5 * (2 ** attempt))
    return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2), "error": err}


print("llm client ready:", FOUNDRY_URL.split("/anthropic")[0], "| model:", settings["model"])


In [ ]:
# [c09] io_load
# 讀 silver，把一個工作項（公司 × 期別）需要的列 collect 到 driver 組 facts。每家公司：financials ≤ 幾十列、文件 ≤ 幾份，
# collect 可接受；文件全文（text_md）只取該公司該期、最多 max_total_chars 字進 prompt。


def table_name(settings: dict, layer: str, short: str) -> str:
    return f"{settings['catalog']}.{settings['schema']}.{LAYER_PREFIX[layer]}{settings['domain']}_{short}"


def _rows(df: DataFrame) -> list[dict]:
    return [r.asDict(recursive=True) for r in df.collect()]


def load_company_bundle(settings: dict, work: dict) -> dict:
    """work = select_work 的一列（company_key / period / …）。回 {company, fin_rows, summary, conference, docs}。"""
    ck, period = work["company_key"], work["period"]
    s = settings
    company_rows = _rows(latest_per_key(spark.table(table_name(s, "silver", SRC_COMPANY)).filter(F.col("company_key") == ck),
                                        ["company_key"]))
    company = company_rows[0] if company_rows else {"company_key": ck}
    company.setdefault("company_slug", work.get("company_slug"))
    company.setdefault("category", work.get("category"))
    fin_rows = _rows(spark.table(table_name(s, "silver", SRC_FINANCIALS)).filter(F.col("company_key") == ck))
    sm_rows = _rows(spark.table(table_name(s, "silver", SRC_SUMMARY))
                    .filter((F.col("company_key") == ck) & (F.col("period") == period)))
    conf_rows = _rows(latest_per_key(spark.table(table_name(s, "silver", SRC_CONFERENCE))
                                     .filter((F.col("company_key") == ck) & (F.col("period") == period)),
                                     ["company_key", "period"], order_col="revision"))
    df_text = (spark.table(table_name(s, "silver", SRC_DOC_TEXT))
               .filter((F.col("company_key") == ck) & (F.col("period") == period) & F.col("parse_status").isin(*DOC_STATUS_OK)))
    df_text = latest_per_key(df_text, ["volume_path"], order_col="parsed_at")
    df_file = spark.table(table_name(s, "silver", SRC_DOC_FILE)).select("volume_path", "source_url", "doc_date")
    docs = _rows(df_text.join(df_file, "volume_path", "left")
                 .select("volume_path", "file_name", "doc_kind", "sha256", "source_url", "text_chars", "text_md", "fiscal_label"))
    if conf_rows and conf_rows[0].get("conference_date"):
        conf_rows[0]["conference_date"] = str(conf_rows[0]["conference_date"])
    return {"company": company, "fin_rows": fin_rows, "summary": sm_rows[0] if sm_rows else None,
            "conference": conf_rows[0] if conf_rows else None, "docs": docs}


def build_bundle_facts(bundle: dict, period: str) -> dict:
    return build_facts(bundle["company"], period, bundle["fin_rows"], bundle["summary"], bundle["conference"], bundle["docs"])


In [ ]:
# [c10] io_write
#   write_llm_row  ：bronze b_{domain}_supplier_report_llm，每次呼叫一列 append-only（原始回應 payload 原文，失敗也記）
#   write_report   ：gold  g_{domain}_supplier_report，鍵 (company_key, period) MERGE：同一期重做就覆蓋成最新版
#   write_html     ：Volume <volume_root>/ir_calendar/reports/<分類小寫>/<公司代稱>/<檔名>.html（FUSE 路徑，同名覆蓋）
# 表由 ir_calendar_init_tables [c19a] / [c19b] 建；不存在直接拋錯。dry_run 不會走到這裡。
import os

from delta.tables import DeltaTable
from pyspark.sql.types import (ArrayType, DoubleType, IntegerType, LongType, StringType, StructField, StructType,
                               TimestampType)

LLM_SCHEMA = StructType([
    StructField("company_key", StringType()), StructField("period", StringType()),
    StructField("prompt_version", StringType()), StructField("model", StringType()),
    StructField("attempt", IntegerType()), StructField("docs_digest", StringType()),
    StructField("status", StringType()), StructField("error", StringType()), StructField("stop_reason", StringType()),
    StructField("prompt_chars", IntegerType()), StructField("input_tokens", IntegerType()),
    StructField("output_tokens", IntegerType()), StructField("cache_read_tokens", IntegerType()),
    StructField("latency_s", DoubleType()), StructField("payload", StringType()),
    StructField("called_at", TimestampType()), StructField("job_run_id", StringType()),
])
REPORT_SCHEMA = StructType([
    StructField("company_key", StringType()), StructField("period", StringType()),
    StructField("company_name", StringType()), StructField("english_name", StringType()),
    StructField("stock_code", StringType()), StructField("market", StringType()), StructField("category", StringType()),
    StructField("company_slug", StringType()), StructField("conference_date", StringType()),
    StructField("report_title", StringType()), StructField("template_version", StringType()),
    StructField("prompt_version", StringType()), StructField("model", StringType()),
    StructField("docs_digest", StringType()), StructField("doc_count", IntegerType()),
    StructField("doc_kinds", ArrayType(StringType())), StructField("has_financials", StringType()),
    StructField("annual_periods", IntegerType()), StructField("quarterly_periods", IntegerType()),
    StructField("light", StringType()), StructField("score", IntegerType()),
    StructField("llm_json", StringType()), StructField("html", StringType()), StructField("html_path", StringType()),
    StructField("html_chars", LongType()), StructField("input_tokens", IntegerType()),
    StructField("output_tokens", IntegerType()), StructField("generated_at", TimestampType()),
    StructField("job_run_id", StringType()),
])
REPORT_KEYS = ["company_key", "period"]


def write_llm_row(settings: dict, row: dict) -> None:
    table = table_name(settings, "bronze", LLM_TABLE)
    spark.createDataFrame([row], schema=LLM_SCHEMA).write.format("delta").mode("append").saveAsTable(table)


def write_report(settings: dict, row: dict) -> None:
    table = table_name(settings, "gold", REPORT_TABLE)
    df = spark.createDataFrame([row], schema=REPORT_SCHEMA)
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in REPORT_KEYS)
    (DeltaTable.forName(spark, table).alias("t").merge(df.alias("s"), cond)
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


def html_volume_path(settings: dict, facts: dict) -> str:
    cat = (facts["company"].get("category") or "uncategorized").lower()
    slug = facts["company"].get("company_slug") or facts["company"]["company_key"].replace(":", "_")
    return f"{settings['volume_root']}/{REPORT_DIR}/{cat}/{slug}/{report_file_name(facts)}"


def write_html(path: str, content: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(content)


def report_row(settings: dict, work: dict, facts: dict, llm: dict, html_text: str, html_path: str,
               usage: dict, now: datetime) -> dict:
    c = facts["company"]
    return {
        "company_key": work["company_key"], "period": work["period"],
        "company_name": c.get("company_name"), "english_name": c.get("english_name"), "stock_code": c.get("stock_code"),
        "market": c.get("market"), "category": c.get("category"), "company_slug": c.get("company_slug"),
        "conference_date": facts.get("conference_date"),
        "report_title": f"{c.get('company_name')}（{c.get('stock_code') or c.get('company_key')}）{facts['period_label']} 財報＋法說整合報告",
        "template_version": TEMPLATE_VERSION, "prompt_version": settings["prompt_version"], "model": settings["model"],
        "docs_digest": work["docs_digest"], "doc_count": int(work["doc_count"]), "doc_kinds": list(work["doc_kinds"] or []),
        "has_financials": "true" if facts["has_financials"] else "false",
        "annual_periods": len(facts["annual"]), "quarterly_periods": len(facts["quarterly"]),
        "light": llm["procurement_conclusion"]["light"], "score": llm["procurement_conclusion"]["score"],
        "llm_json": json.dumps(llm, ensure_ascii=False), "html": html_text, "html_path": html_path,
        "html_chars": len(html_text), "input_tokens": int(usage.get("input", 0)), "output_tokens": int(usage.get("output", 0)),
        "generated_at": now, "job_run_id": settings["job_run_id"],
    }


In [ ]:
# [c11] run_once
# 一次執行：工作清單（[c04]）→ 每份報告：撈 silver 組 facts（[c09]）→ 組 prompt、呼叫 LLM（[c05]/[c08]）→ normalize →
# 套版（[c07]）→ 寫 bronze / gold / Volume（[c10]）。報告數受 max_reports 限制，ThreadPool 併發 concurrency 份。
# JSON 失敗：把上一輪輸出當 assistant turn、帶錯誤訊息再試 1 次；被 max_tokens 截斷改附 COMPACT_HINT 重打。端點失敗就記 failed 列、不產報告。
# render_only：不呼叫 LLM，拿 gold 既有 llm_json 重新套版（版型 / 圖表改了用）。
from concurrent.futures import ThreadPoolExecutor, as_completed

assert PROMPT_VERSION_IN_CELL == settings["prompt_version"], (
    f"[c05] prompt 版本 {PROMPT_VERSION_IN_CELL} ≠ widget prompt_version {settings['prompt_version']}")

VERSION_TAG = f"template {TEMPLATE_VERSION} / prompt {PROMPT_VERSION_IN_CELL}"


def _llm_row(work: dict, settings: dict, attempt: int, r: dict, prompt_chars: int, status: str, error: str | None,
             now: datetime) -> dict:
    u = r.get("usage") or {}
    return {"company_key": work["company_key"], "period": work["period"], "prompt_version": settings["prompt_version"],
            "model": settings["model"], "attempt": attempt, "docs_digest": work["docs_digest"], "status": status,
            "error": (error or "")[:2000] or None, "stop_reason": r.get("stop_reason"), "prompt_chars": prompt_chars,
            "input_tokens": int(u.get("input", 0)), "output_tokens": int(u.get("output", 0)),
            "cache_read_tokens": int(u.get("cache_read", 0)), "latency_s": float(r.get("latency_s") or 0.0),
            "payload": r.get("text"), "called_at": now, "job_run_id": settings["job_run_id"]}


def generate_with_llm(settings: dict, work: dict, facts: dict, docs: list[dict]) -> tuple[dict | None, dict, str | None]:
    """回 (normalized_llm_output | None, usage_total, error)。每次呼叫都寫一列 bronze。"""
    packed, skipped = pack_documents(docs, max_doc_chars=settings["max_doc_chars"], max_total_chars=settings["max_total_chars"])
    user_prompt = build_user_prompt(facts, packed, skipped)
    messages = [{"role": "user", "content": user_prompt}]
    usage_tot = {"input": 0, "output": 0}
    r1 = call_llm(messages, SYSTEM_PROMPT, settings["model"])
    for k in usage_tot:
        usage_tot[k] += (r1.get("usage") or {}).get(k, 0)
    if r1["error"]:
        write_llm_row(settings, _llm_row(work, settings, 1, r1, len(user_prompt), "failed", r1["error"], datetime.now(UTC)))
        return None, usage_tot, r1["error"]
    truncated = r1["stop_reason"] == "max_tokens"
    obj, perr = (None, "output truncated at max_tokens") if truncated else parse_llm_json(r1["text"])
    write_llm_row(settings, _llm_row(work, settings, 1, r1, len(user_prompt), "success" if obj else "json_error", perr, datetime.now(UTC)))
    if obj is None:
        if truncated:
            messages = [{"role": "user", "content": user_prompt + COMPACT_HINT}]
        else:
            messages += [{"role": "assistant", "content": (r1["text"] or "").strip() or "(empty)"},
                         {"role": "user", "content": RETRY_HINT.format(err=perr)}]
        r2 = call_llm(messages, SYSTEM_PROMPT, settings["model"])
        for k in usage_tot:
            usage_tot[k] += (r2.get("usage") or {}).get(k, 0)
        if r2["error"]:
            write_llm_row(settings, _llm_row(work, settings, 2, r2, len(user_prompt), "failed", r2["error"], datetime.now(UTC)))
            return None, usage_tot, r2["error"]
        obj, perr = (None, "output truncated at max_tokens (retry)") if r2["stop_reason"] == "max_tokens" else parse_llm_json(r2["text"])
        write_llm_row(settings, _llm_row(work, settings, 2, r2, len(user_prompt), "success" if obj else "json_error", perr, datetime.now(UTC)))
        if obj is None:
            return None, usage_tot, f"LLM JSON 兩次都失敗：{perr}"
    return normalize_report(obj), usage_tot, None


def process_work(settings: dict, work: dict, existing_llm_json: str | None = None) -> dict:
    """一份報告。回 {work, ok, error, html_path, usage}。"""
    now = datetime.now(UTC)
    bundle = load_company_bundle(settings, work)
    facts = build_bundle_facts(bundle, work["period"])
    if settings["render_only"]:
        if not existing_llm_json:
            return {"work": work, "ok": False, "error": "render_only 但 gold 沒有既有 llm_json", "html_path": None, "usage": {}}
        llm, usage, err = normalize_report(json.loads(existing_llm_json)), {}, None
    else:
        llm, usage, err = generate_with_llm(settings, work, facts, bundle["docs"])
    if err:
        return {"work": work, "ok": False, "error": err, "html_path": None, "usage": usage}
    html_text = render_report(facts, llm, generated_on=now.strftime("%Y-%m-%d"), version=VERSION_TAG)
    path = html_volume_path(settings, facts)
    write_html(path, html_text)
    write_report(settings, report_row(settings, work, facts, llm, html_text, path, usage, now))
    return {"work": work, "ok": True, "error": None, "html_path": path, "usage": usage}


def run_once(settings: dict) -> dict:
    df_text = spark.table(table_name(settings, "silver", SRC_DOC_TEXT))
    df_report = spark.table(table_name(settings, "gold", REPORT_TABLE))
    df_work = select_work(df_text, df_report, categories=settings["categories"], company_keys=settings["company_keys"],
                          periods=settings["periods"], prompt_version=settings["prompt_version"],
                          template_version=TEMPLATE_VERSION, force=settings["force_rebuild"] or settings["render_only"])
    work = [r.asDict() for r in df_work.orderBy("category", "company_key", "period").limit(settings["max_reports"]).collect()]
    report = {"work": len(work), "ok": 0, "errors": [], "paths": [], "usage": {"input": 0, "output": 0}}
    print(f"待產報告 {len(work)} 份（max_reports={settings['max_reports']} categories={settings['categories']} "
          f"company_keys={settings['company_keys']} periods={settings['periods']} force={settings['force_rebuild']} "
          f"render_only={settings['render_only']}）")
    for w in work:
        print(f"  {w['category']:<12} {w['company_key']:<12} {w['company_slug'] or '':<24} {w['period']}  {w['doc_count']} 檔 {list(w['doc_kinds'] or [])}")
    if not work:
        return report
    if settings["dry_run"]:
        for w in work:
            bundle = load_company_bundle(settings, w)
            facts = build_bundle_facts(bundle, w["period"])
            packed, skipped = pack_documents(bundle["docs"], max_doc_chars=settings["max_doc_chars"], max_total_chars=settings["max_total_chars"])
            prompt = build_user_prompt(facts, packed, skipped)
            print(f"  [dry_run] {w['company_key']} {w['period']}: financials annual={len(facts['annual'])} quarterly={len(facts['quarterly'])} "
                  f"docs={len(packed)}（截尾 {sum(d['truncated'] for d in packed)}，略過 {skipped}）prompt_chars={len(prompt)} "
                  f"→ {html_volume_path(settings, facts)}")
        return report

    existing = {}
    if settings["render_only"]:
        keys = [(w["company_key"], w["period"]) for w in work]
        for r in df_report.select("company_key", "period", "llm_json").collect():
            if (r["company_key"], r["period"]) in keys:
                existing[(r["company_key"], r["period"])] = r["llm_json"]

    results = []
    with ThreadPoolExecutor(max_workers=settings["concurrency"]) as ex:
        futs = {ex.submit(process_work, settings, w, existing.get((w["company_key"], w["period"]))): w for w in work}
        for fut in as_completed(futs):
            w = futs[fut]
            try:
                res = fut.result()
            except Exception as e:  # noqa: BLE001 - 一份失敗不拖垮整批，記錯誤繼續
                res = {"work": w, "ok": False, "error": repr(e)[:500], "html_path": None, "usage": {}}
            results.append(res)
            tag = "ok    " if res["ok"] else "FAILED"
            print(f"  {tag} {w['company_key']:<12} {w['period']}  {res['html_path'] or res['error']}")
    for res in results:
        if res["ok"]:
            report["ok"] += 1
            report["paths"].append(res["html_path"])
        else:
            report["errors"].append(f"{res['work']['company_key']} {res['work']['period']}: {res['error']}")
        for k in report["usage"]:
            report["usage"][k] += (res.get("usage") or {}).get(k, 0)
    print(f"完成：{report['ok']} / {len(work)} 份；tokens in={report['usage']['input']} out={report['usage']['output']}；失敗 {len(report['errors'])}")
    return report


In [ ]:
# [c20] main
# job 進入點。有失敗的報告且 fail_on_error = true 才 raise（讓 Databricks job 顯示失敗、通知才會發）。
result = run_once(settings)
print(json.dumps({k: v for k, v in result.items() if k != "paths"}, ensure_ascii=False))
for p in result["paths"]:
    print("  ", p)
if result["errors"] and settings["fail_on_error"]:
    raise RuntimeError(f"{len(result['errors'])} 份報告失敗：" + " | ".join(result["errors"])[:2000])


In [ ]:
# [c30] check_target
# 寫入後驗證（開發用）：gold 鍵唯一、最近幾份報告的摘要、HTML 沒有殘留模板待填欄位 {{…}}。
df_g = spark.table(table_name(settings, "gold", REPORT_TABLE))
n_dup = df_g.groupBy("company_key", "period").count().filter("count > 1").count()
assert n_dup == 0, "g_supplier_report (company_key, period) 重複"
recent = (df_g.select("company_key", "period", "company_name", "light", "score", "has_financials", "doc_count",
                      "html_chars", "prompt_version", "template_version", "generated_at",
                      F.col("html").rlike(r"\{\{[^}]+\}\}").alias("has_placeholder"))
          .orderBy(F.col("generated_at").desc()).limit(20))
for r in recent.collect():
    print(f"{r.generated_at:%Y-%m-%d %H:%M}  {r.company_key:<12} {r.period}  {r.company_name or '':<16} {r.light:<6} "
          f"score={r.score}  fin={r.has_financials}  docs={r.doc_count}  html={r.html_chars}  "
          f"{r.template_version}/{r.prompt_version}  placeholder={r.has_placeholder}")
n_bad = recent.filter("has_placeholder").count()
assert n_bad == 0, f"{n_bad} 份報告 HTML 殘留 {{{{…}}}} 待填欄位"
print(f"rows={df_g.count()} dup_keys={n_dup}")
